In [3]:
# Import Required Libraries
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

In [4]:
# Load Dataset
train_path = "../dataset/Archive-3/train"
test_path = "../dataset/Archive-3/test"

In [5]:
print(type(train_path))
print(train_path)

<class 'str'>
../dataset/Archive-3/train


In [7]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    directory=train_path,
    image_size=(224,224),
    batch_size=32
)

Found 28709 files belonging to 7 classes.


In [9]:
# Get Number of Classes
class_names = train_dataset.class_names

print(class_names)

NUM_CLASSES = len(class_names)

print(NUM_CLASSES)

['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
7


In [10]:
# Build Transfer Learning Model
base_model = MobileNetV2(
    input_shape=(224,224,3),
    include_top=False,
    weights="imagenet"
)



9406464/9406464 [==============================] - 3s 0us/step


In [11]:
# Freeze Base Model
base_model.trainable = False

In [12]:
# Add Your Own Classification Layers
model = models.Sequential([

    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.5),

    layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [13]:
# Check Model Architecture
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.00_224 (Func  (None, 7, 7, 1280)        2257984   
 tional)                                                         
                                                                 
 global_average_pooling2d (  (None, 1280)              0         
 GlobalAveragePooling2D)                                         
                                                                 
 dense (Dense)               (None, 128)               163968    
                                                                 
 dropout (Dropout)           (None, 128)               0         
                                                                 
 dense_1 (Dense)             (None, 7)                 903       
                                                                 
Total params: 2422855 (9.24 MB)
Trainable params: 164871

In [14]:
# Compile Model
model.compile(

    optimizer="adam",

    loss="sparse_categorical_crossentropy",

    metrics=["accuracy"]

)

In [15]:
# Save Best Model
checkpoint = ModelCheckpoint(

    "best_model.keras",

    monitor="val_accuracy",

    save_best_only=True,

    mode="max",

    verbose=1

)

In [16]:
# Add Early Stopping
early_stop = EarlyStopping(

    monitor="val_loss",

    patience=5,

    restore_best_weights=True

)

In [26]:
test_path = r"../dataset/Archive-3/test"

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)

Found 7178 files belonging to 7 classes.


In [27]:
# Train the model
history = model.fit(
    train_dataset,
    validation_data=test_dataset,
    epochs=20,
    callbacks=[
        checkpoint,
        early_stop
    ]
)

Epoch 1/20


898/898 [==============================] - ETA: 0s - loss: 1.8162 - accuracy: 0.2493
Epoch 1: val_accuracy improved from -inf to 0.25467, saving model to best_model.keras
898/898 [==============================] - 648s 696ms/step - loss: 1.8162 - accuracy: 0.2493 - val_loss: 1.7536 - val_accuracy: 0.2547
Epoch 2/20
898/898 [==============================] - ETA: 0s - loss: 1.7685 - accuracy: 0.2688
Epoch 2: val_accuracy improved from 0.25467 to 0.27710, saving model to best_model.keras
898/898 [==============================] - 661s 733ms/step - loss: 1.7685 - accuracy: 0.2688 - val_loss: 1.7278 - val_accuracy: 0.2771
Epoch 3/20
898/898 [==============================] - ETA: 0s - loss: 1.7382 - accuracy: 0.2839
Epoch 3: val_accuracy improved from 0.27710 to 0.29953, saving model to best_model.keras
898/898 [==============================] - 520s 575ms/step - loss: 1.7382 - accuracy: 0.2839 - val_loss: 1.6914 - val_accuracy: 0.2995
Epoch 4/20
898/898 [=====================

In [28]:
# Confirm Model Saved
import os

print(os.listdir())

['best_model.keras', 'phase2_data_loading.ipynb', 'phase3_image_preprocessing.ipynb', 'phase4_model_training.ipynb']


In [30]:
# Fine Tuning
base_model.trainable = True

model.compile(

optimizer=tf.keras.optimizers.Adam(1e-5),

loss="sparse_categorical_crossentropy",

metrics=["accuracy"]

)